<a href="https://colab.research.google.com/github/ASHIKAJAN/ABC-staff-company/blob/main/AI_powered_paraphrasing_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers sentencepiece sentence-transformers language-tool-python rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 2.4 MB/s eta 0:00:00


In [2]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu

In [4]:
print("Loading Models...")

tokenizer = AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws")
model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws")

grammar_tool = language_tool_python.LanguageTool('en-US')

similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Models Loaded Successfully!")


Loading Models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models Loaded Successfully!


In [12]:
def paraphrase(text):

    sentence = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        sentence,
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        do_sample=True,
        top_k=120,
        top_p=0.95,
        early_stopping=True,
        num_return_sequences=1
    )

    paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return paraphrased

In [13]:
def grammar_correct(text):

    matches = grammar_tool.check(text)
    corrected = language_tool_python.utils.correct(text, matches)

    return corrected


In [14]:
def evaluate(original, paraphrased):

    print("\nEvaluation Metrics")

    # BLEU Score
    bleu = sentence_bleu(
        [original.split()],
        paraphrased.split()
    )

    # ROUGE
    scorer = rouge_scorer.RougeScorer(
        ['rouge1','rouge2','rougeL'],
        use_stemmer=True
    )

    rouge = scorer.score(original, paraphrased)

    # Semantic Similarity
    emb1 = similarity_model.encode(original, convert_to_tensor=True)
    emb2 = similarity_model.encode(paraphrased, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()

    print("--------------------------------")
    print("BLEU Score :", round(bleu,4))
    print("ROUGE-1 :", round(rouge['rouge1'].fmeasure,4))
    print("ROUGE-2 :", round(rouge['rouge2'].fmeasure,4))
    print("ROUGE-L :", round(rouge['rougeL'].fmeasure,4))
    print("Semantic Similarity :", round(similarity,4))
    print("--------------------------------")


In [15]:
print("\n========== AI PARAPHRASING TOOL ==========\n")

text = input("Artificial Intelligence is transforming the healthcare industry by improving diagnosis, treatment planning, and patient care.:\n")

print("\nGenerating Paraphrase...\n")

paraphrased = paraphrase(text)

corrected = grammar_correct(paraphrased)

print("Original Text\n")
print(text)

print("\nParaphrased Text\n")
print(paraphrased)

print("\nGrammar Corrected Output\n")
print(corrected)

evaluate(text, corrected)


========== AI PARAPHRASING TOOL ==========

Artificial Intelligence is transforming the healthcare industry by improving diagnosis, treatment planning, and patient care.:
Artificial Intelligence is transforming the healthcare industry by improving diagnosis, treatment planning, and patient care.


[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Generating Paraphrase...

Original Text

Artificial Intelligence is transforming the healthcare industry by improving diagnosis, treatment planning, and patient care.

Paraphrased Text

Artificial intelligence is changing healthcare industry by improving diagnosis, treatment planning and patient care .

Grammar Corrected Output

Artificial intelligence is changing healthcare industry by improving diagnosis, treatment planning and patient care.

Evaluation Metrics
--------------------------------
BLEU Score : 0.436
ROUGE-1 : 0.8966
ROUGE-2 : 0.8148
ROUGE-L : 0.8966
Semantic Similarity : 0.9764
--------------------------------
